# Top110

MBB beam topology optimization using the Guest Heaviside projection filter
from Andreassen et al. (2011). The target MATLAB call is
`top110(60,20,0.5,3,1.8,3)`, with reference compliance `189.1405`.

In [ ]:
import jax
import jax.numpy as np

jax.config.update("jax_enable_x64", True)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax_fem import logger
from PIL import Image as PILImage

logger.setLevel("WARNING")

from topax.filter import build_conv_filter
from topax.optimizer import OC
from topax.projection import proj_fn_guest

In [ ]:
'''

2D MBB beam 

See https://link.springer.com/article/10.1007/s00158-010-0594-7

'''

import jax
from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper

from topax.problem import TopOptProblem


class Elasticity(TopOptProblem):

    def custom_init(self):
        self.fe = self.fes[0]
        self.fe.flex_inds = np.arange(len(self.fe.cells))

    def get_tensor_map(self):
        def stress(u_grad, xPhys):
            # material interpolation
            E = 1e-9 + xPhys**3 * (1.0 - 1e-9)
            nu = 0.3
            # Plane strain
            mu = E/(2.*(1.+nu))
            lmbda = E*nu/((1+nu)*(1-2*nu)) 
            # Plane strain -> Plane Stress
            lmbda = 2*mu*lmbda/(lmbda+2*mu) 
            epsilon = 0.5*(u_grad + u_grad.T)
            sigma = lmbda * np.trace(epsilon) * np.eye(self.dim) + 2*mu*epsilon
            return sigma
        return stress

    def set_params(self, params):
        # Override base class method.
        full_params = np.ones((self.fe.num_cells, params.shape[1]))
        full_params = full_params.at[self.fe.flex_inds].set(params)
        thetas = np.repeat(full_params[:, None, :], self.fe.num_quads, axis=1)
        self.full_params = full_params
        self.internal_vars = [thetas]

    def compute_compliance(self, sol):
        return np.sum(self.point_force * sol[self.load_node])


def prep_fem(Nx, Ny, Lx, Ly):

    # Mesh
    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(Nx, Ny, domain_x=Lx, domain_y=Ly)
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type], ele_type)

    # BCs
    def left(point):
        return np.isclose(point[0], 0., atol=1e-5)

    def right_bottom_corner(point):
        return np.logical_and(np.isclose(point[0], Lx, atol=1e-5), 
                              np.isclose(point[1], 0., atol=1e-5))

    def left_middle(point):
        return np.logical_and(np.isclose(point[0], 0., atol=1e-5), 
                              np.isclose(point[1], Ly, atol=1e-5))

    def dirichlet_val(point):
        return 0.

    dirichlet_bc_info = [[left, right_bottom_corner], 
                         [0, 1], [dirichlet_val]*2]
    # Problem
    problem = Elasticity(mesh, vec=2, dim=2, ele_type=ele_type, 
                         dirichlet_bc_info=dirichlet_bc_info)

    point_force = np.array([0., -1.])
    problem.point_force = point_force
    problem.load_node = problem.add_point_load(left_middle, point_force)

    # Differentiable wrapper
    solver_options = {'petsc_solver':{'ksp_type': 'preonly', 'pc_type': 'lu'}}
    fwd_pred = ad_wrapper(problem, 
                          solver_options=solver_options,
                          adjoint_solver_options=solver_options)

    return fwd_pred, problem

In [ ]:
# SETUP
vf = 0.5
rmin = 1.8
use_density_filter = True
use_sensitivity_filter = False

# forward model
Nx, Ny = 60, 20
Lx, Ly = Nx, Ny
fwd_pred, problem = prep_fem(Nx, Ny, Lx, Ly)

# objective function
def J_total(xPhys):
    sol_list = fwd_pred(xPhys)
    compliance = problem.compute_compliance(sol_list[0])
    return compliance

# constraint function
def volume_constraints(xPhys):
    g = np.sum(xPhys) - vf * xPhys.size
    return g

# filters
H, Hs = build_conv_filter(problem, rmin=rmin)

def density_filter(x):
    x_col = x.reshape(-1, 1)
    return ((H @ x_col) / Hs).reshape(x.shape)

def sensitivity_filter(dc, x):
    dc_col = dc.reshape(-1, 1)
    x_col = x.reshape(-1, 1)
    numerator = H @ (dc_col * x_col)
    denominator = Hs * np.maximum(x_col, 1e-3)
    return (numerator / denominator).reshape(dc.shape)

rho_filter = density_filter if use_density_filter else lambda x: x
sens_filter = sensitivity_filter if use_sensitivity_filter else lambda dc, x: dc

# projection
beta = 1.0
proj = lambda x: proj_fn_guest(x, beta=beta)
transform = lambda x: proj(rho_filter(x))

# optimizer
optimizer = OC(move=0.2, damping=0.5)

# initial design
x0 = vf * np.ones((Nx * Ny, 1))

In [ ]:
# OPTIMIZATION LOOP
loop_beta = 0
loop = 0
change = 1
xnew = x0
xTilde = rho_filter(xnew)
xPhys = proj(xTilde)
frames = []
while change > 0.01:
    loop_beta += 1
    loop += 1
    # Be careful!
    if loop_beta == 1 and beta > 1:
        # just to be consistent with top110
        # however, this may need further check
        J, dJ = jax.value_and_grad(J_total)(xPhys)
        c, dc = jax.value_and_grad(volume_constraints)(xPhys)
        dx = jax.vmap(
            jax.grad(
                lambda value, beta=beta: proj_fn_guest(value, beta=beta)
            )
        )(xTilde.reshape(-1)).reshape(xTilde.shape)
        dJ = H @ ((dJ * dx) / Hs)
        dc = H @ ((dc * dx) / Hs)
    else:
        # evaluate
        J, dJ = jax.value_and_grad(lambda x: J_total(transform(x)))(xnew)
        c, dc = jax.value_and_grad(
            lambda x: volume_constraints(transform(x))
        )(xnew)
    # filter
    xold = xnew.copy()
    dJ = sens_filter(dJ, xold)
    # update
    volume_fn = lambda x: np.mean(transform(x))
    xnew = optimizer.update(xold, dJ, dc, volume_fn, vf)
    xTilde = rho_filter(xnew)
    xPhys = proj(xTilde)
    vol = np.mean(xPhys)
    change = np.max(np.abs(xnew - xold))
    print(f' It.:{loop:5d}, Obj.:{J:11.4f}, Vol.:{vol:7.3f}, ch.:{change:7.3f}')
    # image
    field = onp.flip(xPhys.reshape(Ny, Nx, order='F'), axis=0)
    frames.append(onp.asarray(field))
    # update projection
    if beta < 512 and (loop_beta >= 50 or change <= 0.01):
        xTilde = rho_filter(xnew)
        xPhys = proj(xTilde)
        beta = 2 * beta
        loop_beta = 0
        change = 1
        print(f' Parameter beta increased to {beta}')

In [ ]:
# SAVE OPTIMIZATION HISTORY
output_path = Path("docs/imgs/example_top110.gif")
output_path.parent.mkdir(parents=True, exist_ok=True)

gif_frames = []
for field in frames:
    rgba = plt.get_cmap("gray_r")(onp.clip(field, 0.0, 1.0), bytes=True)
    image = PILImage.fromarray(rgba)
    image = image.resize((Nx * 8, Ny * 8), PILImage.Resampling.NEAREST)
    gif_frames.append(image)

gif_frames[0].save(
    output_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
)
display(DisplayImage(filename=str(output_path)))